In [8]:
import numpy as np
import pandas as pd
from mc_engine.calibration.forward_calibration import *
from mc_engine.processes.hjm import*
from mc_engine.market.market_data import FIMarketData
from mc_engine.processes.hjm import HJMProcess
from mc_engine.products.fixed_income.bond_option import BondOption
from mc_engine.market.curves import YieldCurve
from mc_engine.pricing.engine import MonteCarloEngine
from mc_engine.pricing.config import MCConfig

In [ ]:
df = pd.read_csv("examples/data/hjm.csv",sep="	",index_col=0)/100+1
mkt_data = FIMarketData(r_spot=0.03,r_forward=df.loc[1].to_numpy(),forward_tenors = df.columns)
vol_components = HJMCalbirator(df).create_vol_components()

# Marktdaten
curve = YieldCurve(
    tenors = np.array([0.25, 0.5, 1.0, 2.0, 5.0]),
    rates  = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
)

In [10]:
process = HJMProcess(mkt_data, HJMCalbirator(df).create_vol_components())
option  = BondOption(
    underlying = "Short Rate",
    notional= 100,
    maturity_option   = 1.0,
    maturity_bond = 2.0,
    strike = 0
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=2000, n_steps=252))
result = engine.price(option, {"Short Rate": process}, curve)
print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     13.1070
Std Error: 0.0001
95% CI:    [13.1069, 13.1072]
